# 🛡️ MindGuard — Crisis Detection Evaluation

This notebook performs a comprehensive evaluation of the crisis detection system:
1. Load VAE reconstruction errors and threshold data
2. Compare VAE anomaly detector vs keyword baseline
3. Analyze the dual detection (VAE ∪ Keyword) safety policy
4. Generate publication-quality figures for the report

### Figures generated
- Reconstruction error distribution (crisis vs non-crisis)
- VAE vs Keyword metrics comparison
- Threshold sensitivity analysis (ROC-like curve)
- Confusion matrices for all 3 methods
- Dual detection Venn analysis
- Error percentile analysis

## 1. Setup & Load Data

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
EVAL_DIR = Path("../data/processed/evaluation")
ERRORS_PATH = Path("../models/vae_crisis/reconstruction_errors.csv")
SUMMARY_PATH = Path("../data/processed/vae_threshold_summary.json")
OUTPUT_DIR = EVAL_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Style
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.facecolor": "white",
})

print("✅ Setup complete")

In [ ]:
# Load reconstruction errors
df = pd.read_csv(ERRORS_PATH)
print(f"Total samples: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df["is_crisis"].value_counts().rename({0: "Non-crisis", 1: "Crisis"}))

# Load threshold summary
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
threshold = summary["threshold"]["value"]
vae_metrics = summary["metrics"]
kw_metrics = summary["keyword_baseline_metrics"]

print(f"\nVAE threshold: {threshold:.6f}")
print(f"VAE — P: {vae_metrics['precision']:.4f}, R: {vae_metrics['recall']:.4f}, F1: {vae_metrics['f1']:.4f}")
print(f"KW  — P: {kw_metrics['precision']:.4f}, R: {kw_metrics['recall']:.4f}, F1: {kw_metrics['f1']:.4f}")

## 2. Add Keyword Baseline Predictions

In [ ]:
CRISIS_MARKERS = {
    "suicide", "kill myself", "end my life", "self harm",
    "can't go on", "cannot go on", "want to die", "no reason to live",
}

df["keyword_predicted"] = df["text"].apply(
    lambda t: int(any(m in str(t).lower() for m in CRISIS_MARKERS))
)

# Dual detection: crisis if EITHER VAE or keyword triggers
df["combined_predicted"] = ((df["predicted_crisis"] == 1) | (df["keyword_predicted"] == 1)).astype(int)

y_true = df["is_crisis"].to_numpy()
y_vae = df["predicted_crisis"].to_numpy()
y_kw = df["keyword_predicted"].to_numpy()
y_combined = df["combined_predicted"].to_numpy()
errors = df["reconstruction_error"].to_numpy()

print("Prediction counts:")
print(f"  VAE flagged:      {y_vae.sum():,}")
print(f"  Keyword flagged:  {y_kw.sum():,}")
print(f"  Combined flagged: {y_combined.sum():,}")

## 3. Full Metrics Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def compute_metrics(y_true, y_pred, method_name):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    return {
        "Method": method_name,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }

results = pd.DataFrame([
    compute_metrics(y_true, y_vae, "VAE (Anomaly)"),
    compute_metrics(y_true, y_kw, "Keyword Baseline"),
    compute_metrics(y_true, y_combined, "Combined (VAE ∪ Keyword)"),
])
results[["Precision", "Recall", "F1", "Accuracy"]] = results[["Precision", "Recall", "F1", "Accuracy"]].round(4)

print("\n📊 Crisis Detection — Full Comparison:")
display(results)

results.to_csv(OUTPUT_DIR / "crisis_full_comparison.csv", index=False)
print(f"\n💾 Saved: {OUTPUT_DIR / 'crisis_full_comparison.csv'}")

## 4. Figures

### Figure 1: Reconstruction Error Distribution (Crisis vs Non-Crisis)

In [ ]:
crisis_errors = errors[y_true == 1]
non_crisis_errors = errors[y_true == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(non_crisis_errors, bins=80, alpha=0.7, label="Non-crisis",
             color="#4CAF50", density=True, edgecolor="white", linewidth=0.3)
axes[0].hist(crisis_errors, bins=80, alpha=0.7, label="Crisis",
             color="#F44336", density=True, edgecolor="white", linewidth=0.3)
axes[0].axvline(threshold, color="#FF9800", linestyle="--", linewidth=2,
                label=f"Threshold = {threshold:.5f}")
axes[0].set_xlabel("Reconstruction Error (MSE)")
axes[0].set_ylabel("Density")
axes[0].set_title("Full Error Distribution")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.2)

# Log-scale for better tail visibility
axes[1].hist(non_crisis_errors, bins=80, alpha=0.7, label="Non-crisis",
             color="#4CAF50", density=True, edgecolor="white", linewidth=0.3)
axes[1].hist(crisis_errors, bins=80, alpha=0.7, label="Crisis",
             color="#F44336", density=True, edgecolor="white", linewidth=0.3)
axes[1].axvline(threshold, color="#FF9800", linestyle="--", linewidth=2,
                label=f"Threshold = {threshold:.5f}")
axes[1].set_xlabel("Reconstruction Error (MSE)")
axes[1].set_ylabel("Density (log scale)")
axes[1].set_title("Error Distribution (Log Scale)")
axes[1].set_yscale("log")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.2)

fig.suptitle("VAE Reconstruction Error — Crisis vs Non-Crisis", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "crisis_error_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {OUTPUT_DIR / 'crisis_error_distribution.png'}")

### Figure 2: Three-Method Comparison (Grouped Bar Chart)

In [ ]:
methods = ["VAE\n(Anomaly)", "Keyword\nBaseline", "Combined\n(VAE ∪ KW)"]
precision_vals = results["Precision"].tolist()
recall_vals = results["Recall"].tolist()
f1_vals = results["F1"].tolist()

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(methods))
width = 0.22

bars1 = ax.bar(x - width, precision_vals, width, label="Precision", color="#2196F3", edgecolor="white")
bars2 = ax.bar(x, recall_vals, width, label="Recall", color="#4CAF50", edgecolor="white")
bars3 = ax.bar(x + width, f1_vals, width, label="F1", color="#FF9800", edgecolor="white")

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f"{height:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("Crisis Detection — Method Comparison")
ax.legend(fontsize=10, loc="upper right")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "crisis_method_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {OUTPUT_DIR / 'crisis_method_comparison.png'}")

### Figure 3: Confusion Matrices (All 3 Methods Side by Side)

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ["VAE (Anomaly)", "Keyword Baseline", "Combined (VAE ∪ Keyword)"]
preds_list = [y_vae, y_kw, y_combined]
cmaps = ["Blues", "Greens", "Oranges"]

for ax, title, y_p, cmap in zip(axes, titles, preds_list, cmaps):
    cm = confusion_matrix(y_true, y_p)
    # Normalize to show rates
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=["Non-crisis", "Crisis"],
                yticklabels=["Non-crisis", "Crisis"],
                linewidths=1, linecolor="white", square=True,
                cbar_kws={"shrink": 0.7})

    # Add percentages as secondary annotations
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72, f"({cm_norm[i, j]:.1%})",
                    ha="center", va="center", fontsize=8, color="gray")

    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title, fontsize=11)

fig.suptitle("Confusion Matrices — Crisis Detection Methods", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "crisis_confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {OUTPUT_DIR / 'crisis_confusion_matrices.png'}")

### Figure 4: VAE Threshold Sensitivity Analysis

How do precision, recall, and F1 change as we sweep the threshold?

In [ ]:
# Generate thresholds from error distribution
percentiles = np.arange(50, 100, 1)
all_nc_errors = errors[y_true == 0]  # non-crisis errors for percentile basis

sweep_results = []
for pct in percentiles:
    t = np.percentile(all_nc_errors, pct)
    preds = (errors >= t).astype(int)
    tp = int(((y_true == 1) & (preds == 1)).sum())
    fp = int(((y_true == 0) & (preds == 1)).sum())
    fn = int(((y_true == 1) & (preds == 0)).sum())
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    sweep_results.append({"percentile": pct, "threshold": t, "precision": p, "recall": r, "f1": f})

sweep_df = pd.DataFrame(sweep_results)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sweep_df["percentile"], sweep_df["precision"], label="Precision",
        linewidth=2, color="#2196F3", marker="", linestyle="-")
ax.plot(sweep_df["percentile"], sweep_df["recall"], label="Recall",
        linewidth=2, color="#4CAF50", marker="", linestyle="-")
ax.plot(sweep_df["percentile"], sweep_df["f1"], label="F1",
        linewidth=2.5, color="#FF9800", marker="", linestyle="-")

# Mark the chosen threshold
chosen_pct = summary["threshold"]["percentile"]
ax.axvline(chosen_pct, color="#E53935", linestyle="--", linewidth=1.5,
           label=f"Selected: p{chosen_pct}")

ax.set_xlabel("Threshold Percentile (of non-crisis validation errors)")
ax.set_ylabel("Score")
ax.set_title("VAE Threshold Sensitivity — Precision / Recall / F1 Trade-off")
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=10)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "crisis_threshold_sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {OUTPUT_DIR / 'crisis_threshold_sensitivity.png'}")

### Figure 5: Dual Detection Analysis (Venn-Style)

How do VAE and keyword detections overlap? Does combining them improve safety?

In [ ]:
# Crisis samples detected by each method
crisis_mask = y_true == 1
vae_caught = set(np.where((crisis_mask) & (y_vae == 1))[0])
kw_caught = set(np.where((crisis_mask) & (y_kw == 1))[0])
total_crisis = int(crisis_mask.sum())

both = vae_caught & kw_caught
vae_only = vae_caught - kw_caught
kw_only = kw_caught - vae_caught
neither = total_crisis - len(vae_caught | kw_caught)

print(f"Total crisis samples: {total_crisis:,}")
print(f"VAE only caught:     {len(vae_only):,}")
print(f"Keyword only caught: {len(kw_only):,}")
print(f"Both caught:         {len(both):,}")
print(f"Neither caught:      {neither:,}")

# Stacked bar chart showing detection coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Detection breakdown
categories = ["VAE Only", "Keyword Only", "Both", "Missed"]
values = [len(vae_only), len(kw_only), len(both), neither]
colors = ["#2196F3", "#FF9800", "#4CAF50", "#F44336"]

bars = axes[0].bar(categories, values, color=colors, edgecolor="white", linewidth=1)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
                 f"{val:,}\n({val/total_crisis:.1%})",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[0].set_ylabel("Number of Crisis Samples")
axes[0].set_title("Crisis Detection Coverage Breakdown")
axes[0].grid(axis="y", alpha=0.2)

# Right: Cumulative recall comparison
methods_data = [
    ("VAE Alone", len(vae_caught) / total_crisis),
    ("Keyword Alone", len(kw_caught) / total_crisis),
    ("Combined (VAE ∪ KW)", len(vae_caught | kw_caught) / total_crisis),
]
m_names = [d[0] for d in methods_data]
m_recalls = [d[1] for d in methods_data]
bar_colors = ["#2196F3", "#FF9800", "#4CAF50"]

bars2 = axes[1].bar(m_names, m_recalls, color=bar_colors, edgecolor="white", linewidth=1)
for bar, val in zip(bars2, m_recalls):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f"{val:.1%}", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Recall")
axes[1].set_title("Recall by Detection Method")
axes[1].set_ylim(0, 1.0)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].grid(axis="y", alpha=0.2)

fig.suptitle("Dual Detection System — Safety Coverage Analysis", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "crisis_dual_detection_analysis.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {OUTPUT_DIR / 'crisis_dual_detection_analysis.png'}")

### Figure 6: Error Percentile Analysis (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
box_data = [non_crisis_errors, crisis_errors]
bp = axes[0].boxplot(box_data, labels=["Non-crisis", "Crisis"],
                     patch_artist=True, widths=0.5,
                     medianprops={"color": "black", "linewidth": 2})
bp["boxes"][0].set_facecolor("#4CAF50")
bp["boxes"][0].set_alpha(0.7)
bp["boxes"][1].set_facecolor("#F44336")
bp["boxes"][1].set_alpha(0.7)
axes[0].axhline(threshold, color="#FF9800", linestyle="--", linewidth=1.5,
                label=f"Threshold = {threshold:.5f}")
axes[0].set_ylabel("Reconstruction Error")
axes[0].set_title("Error Distribution by Class")
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.2)

# Violin plot for richer density view
plot_df = pd.DataFrame({
    "Reconstruction Error": errors,
    "Class": ["Crisis" if c == 1 else "Non-crisis" for c in y_true]
})
sns.violinplot(data=plot_df, x="Class", y="Reconstruction Error",
               palette={"Non-crisis": "#4CAF50", "Crisis": "#F44336"},
               inner="quartile", ax=axes[1], cut=0)
axes[1].axhline(threshold, color="#FF9800", linestyle="--", linewidth=1.5,
                label=f"Threshold = {threshold:.5f}")
axes[1].set_title("Error Density by Class (Violin Plot)")
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.2)

fig.suptitle("Reconstruction Error Analysis", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "crisis_error_analysis.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {OUTPUT_DIR / 'crisis_error_analysis.png'}")

### Figure 7: Error Statistics Table

In [ ]:
stats_data = []
for label, err_vals in [("Non-crisis", non_crisis_errors), ("Crisis", crisis_errors)]:
    stats_data.append({
        "Class": label,
        "Count": len(err_vals),
        "Mean": f"{np.mean(err_vals):.6f}",
        "Std": f"{np.std(err_vals):.6f}",
        "Median": f"{np.median(err_vals):.6f}",
        "P25": f"{np.percentile(err_vals, 25):.6f}",
        "P75": f"{np.percentile(err_vals, 75):.6f}",
        "P95": f"{np.percentile(err_vals, 95):.6f}",
        "Max": f"{np.max(err_vals):.6f}",
    })

stats_df = pd.DataFrame(stats_data)
print("\n📊 Reconstruction Error Statistics:")
display(stats_df)

stats_df.to_csv(OUTPUT_DIR / "crisis_error_statistics.csv", index=False)
print(f"\n💾 Saved: {OUTPUT_DIR / 'crisis_error_statistics.csv'}")

## 5. Discussion & Interpretation

### Key Findings

1. **VAE standalone performance is low** (recall ~3.2%) — the reconstruction error distributions for crisis and non-crisis text overlap heavily. This is because TF-IDF features capture vocabulary presence, not semantic meaning. Crisis text like *"I feel like there's no way out"* uses common words that appear in non-crisis text too.

2. **Keyword baseline outperforms VAE** (F1: 0.501 vs 0.061) — simple lexical matching catches explicit crisis language effectively. However, it misses indirect or implicit crisis expressions.

3. **Combined detection improves recall** — the union of both methods catches more crisis cases than either alone, at the cost of some precision. This is the right trade-off for a safety-critical system.

4. **Threshold sensitivity analysis** shows that lowering the percentile threshold increases recall but at rapidly decreasing precision — confirming the chosen p95 threshold is reasonable.

### Implications for the Report

- **Honest evaluation**: Don't hide the VAE's poor standalone recall — present it as a finding about the limitations of TF-IDF-based anomaly detection for semantic tasks
- **Safety argument**: The dual detection approach is a deliberate design choice. Frame it as responsible AI engineering
- **Future work**: Suggest using sentence-level embeddings (e.g., BERT CLS token) instead of TF-IDF for the VAE input, which would likely improve semantic separation

### For the Report — Recommended Figures
- **Figure 1** (error distribution): shows why VAE struggles — distributions overlap
- **Figure 2** (method comparison): clear headline result
- **Figure 3** (confusion matrices): detailed classification performance
- **Figure 5** (dual detection): justifies the safety-first design decision

In [ ]:
# Save complete evaluation summary
eval_summary = {
    "threshold": threshold,
    "threshold_percentile": summary["threshold"]["percentile"],
    "vae_metrics": {k: float(v) for k, v in compute_metrics(y_true, y_vae, "VAE").items() if isinstance(v, (int, float))},
    "keyword_metrics": {k: float(v) for k, v in compute_metrics(y_true, y_kw, "Keyword").items() if isinstance(v, (int, float))},
    "combined_metrics": {k: float(v) for k, v in compute_metrics(y_true, y_combined, "Combined").items() if isinstance(v, (int, float))},
    "detection_overlap": {
        "vae_only": len(vae_only),
        "keyword_only": len(kw_only),
        "both": len(both),
        "neither": neither,
        "total_crisis": total_crisis,
    },
    "figures_saved": [
        "crisis_error_distribution.png",
        "crisis_method_comparison.png",
        "crisis_confusion_matrices.png",
        "crisis_threshold_sensitivity.png",
        "crisis_dual_detection_analysis.png",
        "crisis_error_analysis.png",
    ]
}
(OUTPUT_DIR / "crisis_full_evaluation.json").write_text(
    json.dumps(eval_summary, indent=2)
)

print("\n" + "=" * 60)
print("🛡️ CRISIS EVALUATION COMPLETE")
print("=" * 60)
for m in ["VAE", "Keyword", "Combined"]:
    row = results[results["Method"].str.contains(m)].iloc[0]
    print(f"{m:12s} — P: {row['Precision']:.4f}, R: {row['Recall']:.4f}, F1: {row['F1']:.4f}")
print(f"\n6 figures + 2 CSVs + 1 JSON saved to: {OUTPUT_DIR}")
print("=" * 60)